In [ ]:
# Installing my gpu library donot add gpu library to poetry as it is specific to machine
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (6.3 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torchaudio-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (6.8 kB)
  Using cached https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (28 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached https://download-r2.pytorch.org/whl/cu124/torchvision-0.21.0%2Bcu124-cp311-cp311-win_amd64.whl (6.1 MB)
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 2.2 MB/s eta 0:18:50
   ---------------------------------------- 0.0/2.5 GB 2.5 MB/s eta 0:16:59
   ---------------------------------------- 0.0/2.5 GB 2.8 MB/s eta 0:15:05
   -------------------

## Cell 1 - Imports and setup

In [3]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from loguru import logger

# config.py lives at ../assignment3/config.py relative to this file.
try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent / "assignment3"))

from config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, FIGURES_DIR

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# -- Device detection ------------------------------------------------------
# Auto-pick CUDA > MPS (Apple Silicon) > CPU. Batch size scales with device
# so CPU runs don't thrash memory and GPU runs actually saturate the card.
if torch.cuda.is_available():
    DEVICE = "cuda"
    EMBED_BATCH_SIZE = 256
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem_gb:.1f} GB)")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    EMBED_BATCH_SIZE = 128
    print("Apple Silicon MPS detected")
else:
    DEVICE = "cpu"
    EMBED_BATCH_SIZE = 32
    print("No GPU detected — running on CPU (slower but works fine)")

print(f"Device: {DEVICE} | Embed batch size: {EMBED_BATCH_SIZE}")

# Vector store location — persistent so teammates don't have to re-embed
VECTOR_STORE_DIR = PROCESSED_DATA_DIR / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Interim dir:    {INTERIM_DATA_DIR}")
print(f"Processed dir:  {PROCESSED_DATA_DIR}")
print(f"Vector store:   {VECTOR_STORE_DIR}")

2026-04-24 14:57:30.657 | INFO     | config:<module>:11 - PROJ_ROOT path is: D:\MS_DSI\AI\Assignment3\assignment3


GPU detected: NVIDIA GeForce GTX 1650 (4.3 GB)
Device: cuda | Embed batch size: 256
Interim dir:    D:\MS_DSI\AI\Assignment3\assignment3\data\interim
Processed dir:  D:\MS_DSI\AI\Assignment3\assignment3\data\processed
Vector store:   D:\MS_DSI\AI\Assignment3\assignment3\data\processed\vector_store


# PART 1 - LOAD AND PREPARE DOCUMENTS

## Cell 2 - Load cleaned Counsel Chat

In [4]:
counsel = pd.read_csv(INTERIM_DATA_DIR / "counsel_chat_clean.csv")
print(f"Loaded {len(counsel):,} cleaned Q&A rows")
print(f"Columns: {counsel.columns.tolist()}")
counsel.head(3)

Loaded 2,599 cleaned Q&A rows
Columns: ['topic', 'question_clean', 'answer_clean', 'a_words_clean']


,topic,question_clean,answer_clean,a_words_clean
0,depression,"I have so many issues to address. I have a history of sexual abuse, I m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I m beginning to have anxiet...","It is very common for people to have multiple issues that they want to (and need to) address in counseling. I have had clients ask that same question and through more exploration, there is often a...",172
1,depression,"I have so many issues to address. I have a history of sexual abuse, I m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I m beginning to have anxiet...","I've never heard of someone having ""too many issues"" for therapy to be effective. A competent therapist will assist you in identifying the root causes of your problems and treat those first. If th...",107
2,depression,"I have so many issues to address. I have a history of sexual abuse, I m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I m beginning to have anxiet...",Absolutely not. I strongly recommending working on one issue/need at a time. In therapy you will set smart goals and objectives that will help you reach your goals. I see you as a survivor and not...,42


## Cell 3 - Decide on chunking strategy

Most answers fit in one chunk (median ~134 words), but the tail goes
up to ~1000. Embedding models truncate at ~256-512 tokens anyway, so
very long answers get cropped silently unless we split them.
Strategy: split any answer longer than CHUNK_MAX_WORDS into overlapping
windows. Short/medium answers pass through unchanged.

In [5]:
CHUNK_MAX_WORDS = 250   # roughly ~330 tokens, safely under 512
CHUNK_OVERLAP_WORDS = 50

long_answer_mask = counsel["a_words_clean"] > CHUNK_MAX_WORDS
print(f"Answers needing split:     {long_answer_mask.sum():,} / {len(counsel):,} "
      f"({long_answer_mask.mean()*100:.1f}%)")
print(f"Average length of long ones: {counsel.loc[long_answer_mask, 'a_words_clean'].mean():.0f} words")

Answers needing split:     487 / 2,599 (18.7%)
Average length of long ones: 372 words


## Cell 4 - Chunk long answers into overlapping windows

We chunk ONLY the answer, but keep the question attached to every
chunk so retrieval works on either side of the Q+A pair.

In [6]:
def split_into_chunks(text: str, max_words: int, overlap: int):
    """Split `text` into word-windows of size `max_words` with `overlap`."""
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        piece = " ".join(words[start:start + max_words])
        chunks.append(piece)
        if start + max_words >= len(words):
            break
    return chunks


# Build the long-form chunked dataframe.
records = []
for idx, row in counsel.iterrows():
    chunks = split_into_chunks(row["answer_clean"], CHUNK_MAX_WORDS, CHUNK_OVERLAP_WORDS)
    for chunk_ix, chunk in enumerate(chunks):
        records.append({
            "chunk_id":    f"row{idx}_c{chunk_ix}",
            "row_id":      idx,
            "topic":       row.get("topic", ""),
            "question":    row["question_clean"],
            "answer":      row["answer_clean"],
            "chunk_text":  chunk,
            "chunk_words": len(chunk.split()),
            "n_chunks":    len(chunks),
        })

docs = pd.DataFrame(records)
print(f"Total chunks: {len(docs):,}  (from {len(counsel):,} Q&A rows)")
print(f"Expansion factor: {len(docs) / len(counsel):.2f}x")
print(f"\nChunk length stats:")
print(docs["chunk_words"].describe().round(1))

Total chunks: 3,193  (from 2,599 Q&A rows)
Expansion factor: 1.23x

Chunk length stats:
count    3193.0
mean      148.1
std        70.4
min        15.0
25%        88.0
50%       138.0
75%       215.0
max       250.0
Name: chunk_words, dtype: float64


## Cell 5 - Build the text we'll actually embed (Q + A combined)

Putting the question in front of the chunk gives the retriever both
"concern language" and "advice language" to match on. Questions are
duplicated across chunks of the same answer, which is fine — it
reinforces the topic signal.

In [7]:
def build_embedding_text(question: str, chunk: str) -> str:
    return f"Question: {question}\n\nAnswer: {chunk}"


docs["embed_text"] = docs.apply(
    lambda r: build_embedding_text(r["question"], r["chunk_text"]), axis=1
)

print("Sample of what gets embedded:\n")
print("=" * 80)
print(docs["embed_text"].iloc[0][:600])
print("=" * 80)

Sample of what gets embedded:

Question: I have so many issues to address. I have a history of sexual abuse, I m a breast cancer survivor and I am a lifetime insomniac. I have a long history of depression and I m beginning to have anxiety. I have low self esteem but I ve been happily married for almost 35 years. I ve never had counseling about any of this. Do I have too many issues to address in counseling?

Answer: It is very common for people to have multiple issues that they want to (and need to) address in counseling. I have had clients ask that same question and through more exploration, there is often an underlying fe


# PART 2 - EMBED AND INDEX WITH CHROMADB

## Cell 6 - Load the embedding model

MiniLM-L6-v2: 22M params, 384-dim, ~80MB, fast on CPU.
Standard strong baseline for English semantic search.

In [ ]:
from sentence_transformers import SentenceTransformer

PRIMARY_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

logger.info(f"Loading {PRIMARY_MODEL} on {DEVICE}...")
embedder = SentenceTransformer(PRIMARY_MODEL, device=DEVICE)
print(f"Model loaded. Embedding dim: {embedder.get_sentence_embedding_dimension()}")

e:\poetry_cache\virtualenvs\assignment3-HMuFVmZG-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 7 - Embed all chunks (one-off, ~1-2 min on CPU)

normalize_embeddings=True so cosine similarity works as a dot product
when we query later.

In [ ]:
import time

logger.info(f"Embedding {len(docs):,} chunks on {DEVICE}...")
t0 = time.time()
embeddings = embedder.encode(
    docs["embed_text"].tolist(),
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"Embeddings shape: {embeddings.shape}")
print(f"Elapsed: {elapsed:.1f}s  ({len(docs) / elapsed:.1f} chunks/sec)")

## Cell 8 - Build the ChromaDB collection

Persistent client writes to disk so we don't re-embed on every run.
If the collection already exists we delete and rebuild — simpler than
detecting stale chunks. For a 2k-doc corpus this rebuild is seconds.

In [ ]:
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path=str(VECTOR_STORE_DIR),
    settings=Settings(anonymized_telemetry=False),
)

COLLECTION_NAME = "counsel_chat_minilm"

# Rebuild cleanly
try:
    client.delete_collection(COLLECTION_NAME)
    logger.info(f"Dropped existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"embedder": PRIMARY_MODEL, "dim": embeddings.shape[1]},
)

# ChromaDB requires string ids. Metadata must be primitive types.
collection.add(
    ids=docs["chunk_id"].tolist(),
    embeddings=embeddings.tolist(),
    documents=docs["embed_text"].tolist(),
    metadatas=[
        {
            "topic":   str(r["topic"]) if pd.notna(r["topic"]) else "",
            "row_id":  int(r["row_id"]),
            "question": r["question"][:500],  # Chroma metadata has size limits
            "answer":   r["answer"][:500],
            "chunk_words": int(r["chunk_words"]),
        }
        for _, r in docs.iterrows()
    ],
)
print(f"Indexed {collection.count():,} chunks into collection '{COLLECTION_NAME}'")

# PART 3 - RETRIEVAL API + SANITY CHECK

## Cell 9 - Wrap retrieval in a clean function

This is the function downstream team members (Team B) will import
when they build the LLM prompt-assembly layer.

In [ ]:
def retrieve(query: str, k: int = 5, topic_filter: str = None):
    """
    Retrieve top-k chunks for a query.

    Args:
        query: user message, raw text
        k: number of chunks to return
        topic_filter: optional topic to constrain retrieval (e.g., 'anxiety')

    Returns a pandas DataFrame with columns:
        rank, similarity, topic, question, answer_snippet, chunk_text
    """
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()

    where = {"topic": topic_filter} if topic_filter else None
    result = collection.query(
        query_embeddings=q_emb,
        n_results=k,
        where=where,
    )

    # Chroma returns cosine *distance*; convert to similarity for readability
    distances = result["distances"][0]
    similarities = [1 - d for d in distances]

    rows = []
    for rank, (meta, doc, sim) in enumerate(
        zip(result["metadatas"][0], result["documents"][0], similarities), 1
    ):
        rows.append({
            "rank": rank,
            "similarity": round(sim, 3),
            "topic": meta.get("topic", ""),
            "question": meta.get("question", ""),
            "answer_snippet": meta.get("answer", "")[:200] + "...",
            "chunk_text": doc,
        })
    return pd.DataFrame(rows)

## Cell 10 - Sanity check with real-sounding queries

If these don't return sensible results, the index is broken or the
embedding model is a bad fit. Don't move on to Member 4's LLM step
until these look good.

In [ ]:
SANITY_QUERIES = [
    "I've been feeling really anxious about my upcoming exams",
    "my partner cheated on me and I don't know if I can forgive them",
    "how do I deal with my depression when nothing seems to help",
    "I'm struggling to connect with my teenage daughter",
    "I lost my job last month and I can't stop feeling worthless",
]

for q in SANITY_QUERIES:
    print(f"\n{'=' * 80}\nQUERY: {q}\n{'=' * 80}")
    results = retrieve(q, k=3)
    for _, r in results.iterrows():
        print(f"[{r['rank']}] sim={r['similarity']:.3f} topic={r['topic']}")
        print(f"    Q: {r['question'][:150]}")
        print(f"    A: {r['answer_snippet']}")

## Cell 11 - Demo: topic-filtered retrieval

This is a concrete feature we can show on the presentation demo.
"Same query, different topic constraints" proves the metadata layer works.

In [ ]:
demo_query = "I feel like nothing I do is good enough"

for topic in ["depression", "self-esteem", "anxiety"]:
    print(f"\n--- {topic.upper()} ---")
    results = retrieve(demo_query, k=2, topic_filter=topic)
    for _, r in results.iterrows():
        print(f"  sim={r['similarity']:.3f} | Q: {r['question'][:120]}")

# PART 4 - A/B COMPARISON: MINILM vs BGE

## Cell 12 - Build an alternative index with BGE for comparison

BGE-small-en-v1.5 is the current retrieval SOTA in the small-model
bracket (still 384-dim, still fast on CPU). Running both gives us a
justified method-choice slide for the presentation rubric.
First run downloads ~130MB.

In [ ]:
ALT_MODEL = "BAAI/bge-small-en-v1.5"
logger.info(f"Loading alternative model {ALT_MODEL} on {DEVICE}...")
alt_embedder = SentenceTransformer(ALT_MODEL, device=DEVICE)

logger.info(f"Re-embedding with {ALT_MODEL}...")
t0 = time.time()
alt_embeddings = alt_embedder.encode(
    docs["embed_text"].tolist(),
    batch_size=EMBED_BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
)
elapsed = time.time() - t0
print(f"Alt embeddings shape: {alt_embeddings.shape}")
print(f"Elapsed: {elapsed:.1f}s  ({len(docs) / elapsed:.1f} chunks/sec)")

## Cell 13 - Build the BGE collection

In [ ]:
ALT_COLLECTION_NAME = "counsel_chat_bge"

try:
    client.delete_collection(ALT_COLLECTION_NAME)
except Exception:
    pass

alt_collection = client.create_collection(
    name=ALT_COLLECTION_NAME,
    metadata={"embedder": ALT_MODEL, "dim": alt_embeddings.shape[1]},
)

alt_collection.add(
    ids=docs["chunk_id"].tolist(),
    embeddings=alt_embeddings.tolist(),
    documents=docs["embed_text"].tolist(),
    metadatas=[
        {
            "topic":   str(r["topic"]) if pd.notna(r["topic"]) else "",
            "row_id":  int(r["row_id"]),
            "question": r["question"][:500],
            "answer":   r["answer"][:500],
            "chunk_words": int(r["chunk_words"]),
        }
        for _, r in docs.iterrows()
    ],
)
print(f"Indexed {alt_collection.count():,} chunks with {ALT_MODEL}")

## Cell 14 - Side-by-side retrieval comparison

For each sanity query, print top-3 from BOTH models so we can eyeball
agreement. High overlap = both models agree, low overlap = they disagree
and we'd need a larger evaluation (Role 5's job) to judge which is better.

In [ ]:
def retrieve_from(coll, embedder_model, query, k=3):
    q_emb = embedder_model.encode([query], normalize_embeddings=True).tolist()
    res = coll.query(query_embeddings=q_emb, n_results=k)
    return [
        {
            "rank": i + 1,
            "similarity": round(1 - d, 3),
            "question": m.get("question", "")[:120],
        }
        for i, (m, d) in enumerate(zip(res["metadatas"][0], res["distances"][0]))
    ]


overlap_scores = []
for q in SANITY_QUERIES:
    print(f"\n{'=' * 80}\nQUERY: {q}\n{'=' * 80}")

    mini_results = retrieve_from(collection, embedder, q, k=5)
    bge_results = retrieve_from(alt_collection, alt_embedder, q, k=5)

    mini_ids = {r["question"] for r in mini_results}
    bge_ids = {r["question"] for r in bge_results}
    overlap = len(mini_ids & bge_ids) / max(len(mini_ids), 1)
    overlap_scores.append(overlap)

    print(f"\nTop-5 overlap between models: {overlap:.0%}\n")
    print(f"{'MiniLM top-3':<50s} | {'BGE top-3':<50s}")
    print("-" * 105)
    for i in range(3):
        m = mini_results[i]["question"]
        b = bge_results[i]["question"]
        print(f"[{mini_results[i]['similarity']:.2f}] {m[:44]:<44s} | [{bge_results[i]['similarity']:.2f}] {b[:44]:<44s}")

print(f"\n{'=' * 80}")
print(f"Mean top-5 overlap across {len(SANITY_QUERIES)} queries: {np.mean(overlap_scores):.0%}")
print(f"{'=' * 80}")

## Cell 15 - Plot the model-agreement comparison for the slide deck

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(SANITY_QUERIES)), overlap_scores, color="#534AB7")
ax.set_xticks(range(len(SANITY_QUERIES)))
ax.set_xticklabels([q[:30] + "..." for q in SANITY_QUERIES], rotation=20, ha="right")
ax.set_ylabel("Top-5 retrieval overlap")
ax.set_ylim(0, 1)
ax.axhline(np.mean(overlap_scores), color="#D85A30", linestyle="--",
           label=f"mean = {np.mean(overlap_scores):.0%}")
ax.set_title("MiniLM vs BGE — top-5 retrieval agreement per query")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "retrieval_model_agreement.png", dpi=120)
plt.show()

# SUMMARY

Outputs:
- Two ChromaDB collections in data/processed/vector_store/:
'counsel_chat_minilm' (primary, what we'll use in the app)
'counsel_chat_bge'    (alternative, for comparison slide)
- reports/figures/retrieval_model_agreement.png

The `retrieve(query, k, topic_filter=None)` function (Cell 9) is the
public API for this module. Team B imports it to build the prompt.

For the slide deck:
- Cell 10 output: qualitative examples of "it works" (pick 2 best ones)
- Cell 11 output: the topic-filtering demo (concrete feature to show)
- Cell 14 numbers + Cell 15 chart: method-choice justification

NEXT NOTEBOOK (Role 5 evaluation - formal retrieval metrics):
03_retrieval_evaluation.py
- Build 50-query test set with expected-topic labels
- Compute Precision@5, Mean Reciprocal Rank for both models
- Pick a winner based on numbers, not vibes